In [0]:
%run ../00-common/01.environment-config

In [0]:
bronze_table = f"{catlog_name}.{bronze_schema}.circuits"
silver_table = f"{catlog_name}.{silver_schema}.circuits"



In [0]:
circuits_df = spark.table(bronze_table)

In [0]:
display(circuits_df)

In [0]:
circuit_selected_df = circuits_df.select(
    "circuitID",
    "circuitName",
    "lat",
    "long",
    "locality",
    "country",
    "ingestion_timestamp",
    "source_file"
)

In [0]:
from pyspark.sql import functions as F

In [0]:
circuit_selected_df = circuits_df.select(
    F.col("circuitID"),
    F.col("circuitName"),
    F.col("lat"),
    F.col("long"),
    F.col("locality"),
    F.col("country"),
    F.col("ingestion_timestamp"),
    F.col("source_file")
)

In [0]:
circuit_renamed_df = (
    circuit_selected_df
        .withColumnsRenamed({
            "circuitID": "circuit_id",
            "circuitName": "circuit_name",
            "lat": "latitude",
            "long": "longitude",
            
        })
        
)

In [0]:
display(circuit_renamed_df)

In [0]:
circuit_valid_df = circuit_renamed_df.filter(
    F.col("circuit_id").isNotNull()
)

In [0]:
display(circuit_valid_df)

In [0]:
circuit_distinct_df = circuit_valid_df.distinct()

In [0]:
display(circuit_distinct_df)

In [0]:
circuits_final_df = (
    circuit_distinct_df
    .withColumn("circuit_name", F.initcap(F.col("circuit_name")))
    .withColumn("locality", F.initcap(F.col("locality")))
)

In [0]:
display(circuits_final_df)

In [0]:
(
    circuits_final_df
        .write
        .format('delta')
        .mode("overwrite")
        .saveAsTable(silver_table)
)

In [0]:
display(spark.table(silver_table))